In [ ]:
from pathlib import Path
import zlib
import pandas as pd

In [ ]:
FOLDER = Path("NO_STRUCTURE_LLM_CLEANED")
rows = []

for fp in FOLDER.rglob("*.txt"):
    text = fp.read_text(encoding="utf-8")

    raw_bytes = text.encode("utf-8")
    compressed_bytes = zlib.compress(raw_bytes)

    compression_ratio = len(compressed_bytes) / len(raw_bytes) if len(raw_bytes) > 0 else None

    is_bad = (
        compression_ratio is not None
        and compression_ratio < 0.25
        and len(text) > 200)

    rows.append({
        "file": str(fp),
        "chars": len(text),
        "words": len(text.split()),
        "compression_ratio": compression_ratio,
        "is_bad": is_bad})

    if is_bad:
        print("\nBAD FILE:")
        print(fp)
        print(f"Compression ratio: {compression_ratio:.2f}")
        print("-" * 80)

df = pd.DataFrame(rows)
df.to_csv("compression_check_results.csv", index=False)

print(df['is_bad'].sum())
print("Done.")
#checked


BAD FILE:
NO_STRUCTURE_LLM_CLEANED\Scottish_Government_ECU00004982\extracted_ECU00004982 Representation 046 066 Objection-011.txt
Compression ratio: 0.09
--------------------------------------------------------------------------------

BAD FILE:
NO_STRUCTURE_LLM_CLEANED\Scottish_Government_ECU00004982\extracted_ECU00004982 Representation 096 116 Objection-001.txt
Compression ratio: 0.09
--------------------------------------------------------------------------------

BAD FILE:
NO_STRUCTURE_LLM_CLEANED\Scottish_Government_ECU00004982\extracted_ECU00004982 Representation 138 155 Objection-008.txt
Compression ratio: 0.08
--------------------------------------------------------------------------------

BAD FILE:
NO_STRUCTURE_LLM_CLEANED\Scottish_Government_ECU00004982\extracted_ECU00004982 Representation 238 - Objection.txt
Compression ratio: 0.08
--------------------------------------------------------------------------------

BAD FILE:
NO_STRUCTURE_LLM_CLEANED\Scottish_Government_ECU000

In [ ]:
INPUT_FOLDER = Path("C:/Users/wgarl/OneDrive/Pulpit/thesis/_SCOTBESS_DATA_CORRECTED/JSON_NO_STRUCTURE_EXTRACTED_TXT")
OUTPUT_FOLDER = Path("NO_STRUCTURE_LLM_CLEANED_CHECKED")

MAX_INCREASE = 0.01

rows = []

for output_fp in OUTPUT_FOLDER.rglob("*.txt"):
    output_relative_path = output_fp.relative_to(OUTPUT_FOLDER)

    if not output_fp.name.startswith("extracted_"):
        print(f"Unexpected output filename: {output_fp}")
        continue

    input_filename = output_fp.name.removeprefix("extracted_")

    #the same relative subfolder structure
    input_relative_path = output_relative_path.parent / input_filename
    input_fp = INPUT_FOLDER / input_relative_path

    if not input_fp.exists():
        print(f"Missing corresponding input file: {input_fp}")
        continue

    try:
        input_text = input_fp.read_text(encoding="utf-8")
        output_text = output_fp.read_text(encoding="utf-8")
    except Exception as e:
        print(f"Error reading {output_relative_path}: {e}")
        continue

    input_chars = len(input_text)
    output_chars = len(output_text)

    input_words = len(input_text.split())
    output_words = len(output_text.split())

    if input_words > 0:
        word_length_ratio = output_words / input_words
        word_increase_pct = ((output_words - input_words) / input_words) * 100

        is_too_long = output_words > input_words * (1 + MAX_INCREASE)

    else:
        word_length_ratio = None
        word_increase_pct = None
        is_too_long = output_words > 0

    rows.append({
        "relative_path": str(input_relative_path),
        "input_file": str(input_fp),
        "output_file": str(output_fp),
        "input_chars": input_chars,
        "output_chars": output_chars,
        "input_words": input_words,
        "output_words": output_words,
        "word_difference": output_words - input_words,
        "word_length_ratio": word_length_ratio,
        "word_increase_pct": word_increase_pct,
        "is_more_than_1_percent_longer": is_too_long})

    if is_too_long:
        print("\nOUTPUT MORE THAN 1% LONGER:")
        print(f"Input:  {input_fp}")
        print(f"Output: {output_fp}")
        print(f"Input words:  {input_words}")
        print(f"Output words: {output_words}")

        if word_increase_pct is not None:
            print(f"Increase: {word_increase_pct:.2f}%")

        print("-" * 80)

df = pd.DataFrame(rows)

bad_df = df[df["is_more_than_1_percent_longer"]].copy()

print()
print(f"Successfully compared files: {len(df)}")
print(f"Files more than 1% longer: {len(bad_df)}")
print("Done.")

display(bad_df)
#1 - hallucinated, no body irl, deleted
#2 - good
#3 - hallucinated, no body irl
#4 - slight loop, corrected
#5 - hallucinated by adding reasoning - corrected
#6 hallucinated, no body, deleted


OUTPUT MORE THAN 1% LONGER:
Input:  C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBESS_DATA_CORRECTED\JSON_NO_STRUCTURE_EXTRACTED_TXT\Renfrewshire_24_0153_PP\24_0153_PP-HOUSTON_BOTTLING___CO-PACK-1534626.txt
Output: NO_STRUCTURE_LLM_CLEANED_CHECKED\Renfrewshire_24_0153_PP\extracted_24_0153_PP-HOUSTON_BOTTLING___CO-PACK-1534626.txt
Input words:  110
Output words: 117
Increase: 6.36%
--------------------------------------------------------------------------------

OUTPUT MORE THAN 1% LONGER:
Input:  C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBESS_DATA_CORRECTED\JSON_NO_STRUCTURE_EXTRACTED_TXT\Scottish_Government_ECU00003354\ECU00003354 Representation Redacted 051 054 Objection-006.txt
Output: NO_STRUCTURE_LLM_CLEANED_CHECKED\Scottish_Government_ECU00003354\extracted_ECU00003354 Representation Redacted 051 054 Objection-006.txt
Input words:  96
Output words: 125
Increase: 30.21%
--------------------------------------------------------------------------------

OUTPUT MORE THAN 1% LONGER:
Inp

,relative_path,input_file,output_file,input_chars,output_chars,input_words,output_words,word_difference,word_length_ratio,word_increase_pct,is_more_than_1_percent_longer
162,Renfrewshire_24_0153_PP\24_0153_PP-HOUSTON_BOT...,C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBES...,NO_STRUCTURE_LLM_CLEANED_CHECKED\Renfrewshire_...,796,813,110,117,7,1.063636,6.363636,True
208,Scottish_Government_ECU00003354\ECU00003354 Re...,C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBES...,NO_STRUCTURE_LLM_CLEANED_CHECKED\Scottish_Gove...,633,794,96,125,29,1.302083,30.208333,True
461,Scottish_Government_ECU00004881\ECU00004881 Re...,C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBES...,NO_STRUCTURE_LLM_CLEANED_CHECKED\Scottish_Gove...,1850,1879,255,280,25,1.098039,9.803922,True
1106,Scottish_Government_ECU00005155\ECU00005155 Re...,C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBES...,NO_STRUCTURE_LLM_CLEANED_CHECKED\Scottish_Gove...,11972,16012,1622,2170,548,1.337855,33.785450,True
1999,Scottish_Government_ECU00006053\ECU00006053 Re...,C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBES...,NO_STRUCTURE_LLM_CLEANED_CHECKED\Scottish_Gove...,193,417,28,61,33,2.178571,117.857143,True
2172,Scottish_Government_ECU00006121\ECU00006121 Re...,C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBES...,NO_STRUCTURE_LLM_CLEANED_CHECKED\Scottish_Gove...,344,1695,40,253,213,6.325000,532.500000,True
